# Co-occurrence Language Model

**Course:** Artificial Intelligence · Unit 6 — Natural Language Processing  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 24 — Deep Learning for Natural Language Processing  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Build a word co-occurrence matrix from a Spanish tweet corpus
2. Apply bigram tokenisation and Spanish stopword removal
3. Visualise co-occurrence frequencies as a seaborn heatmap and clustermap
4. Understand how distributional semantics relates words through shared context


---

## 1. Environment Setup

Install once in your terminal:

```bash
pip install nltk pandas numpy matplotlib seaborn
```


In [3]:
# pip install nltk pandas numpy matplotlib seaborn  # run once in terminal
from pathlib import Path

import itertools

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from nltk import bigrams, word_tokenize
from nltk.corpus import stopwords

# Python 3.13 compatibility: punkt_tab replaces the deprecated punkt tokeniser
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

DIR_DATA = Path.cwd() / 'data'


ModuleNotFoundError: No module named 'seaborn'

---

## 2. Load the Corpus

The dataset is a subset of the **TASS 2018** Spanish tweet collection.  
Each row contains a tweet text and a sentiment label.


In [4]:
corpus = pd.read_csv(
    DIR_DATA / 'Valence_test_oc_es.csv',
    sep=';',
    on_bad_lines='skip',
)
print(f'Corpus shape: {corpus.shape}')
print(f'Columns: {corpus.columns.tolist()}')
corpus.head()


NameError: name 'DIR_DATA' is not defined

> **Output interpretation:** The corpus contains Spanish tweets. The 'Tweet' column is the text we will tokenise. Rows with malformed delimiters are skipped (`on_bad_lines='skip'`). Inspect the shape and columns before proceeding to ensure the file loaded correctly.


---

## 3. Build the Co-occurrence Matrix

A **co-occurrence matrix** `M` where `M[i][j]` counts how often word `i` and word `j`  
appear as consecutive bigrams in the corpus. Both orderings are incremented symmetrically.


In [ ]:
SPANISH_STOPWORDS = set(stopwords.words('spanish'))


def generate_co_occurrence_matrix(
    texts: list[str],
) -> tuple[np.ndarray, dict[str, int]]:
    """Build a symmetric word co-occurrence matrix from a list of text strings.

    Words shorter than 4 characters and Spanish stopwords are excluded.
    """
    filtered_corpus = [
        [
            word
            for word in word_tokenize(sentence.lower())
            if word not in SPANISH_STOPWORDS and len(word) > 3
        ]
        for sentence in texts
    ]

    bi_grams = list(itertools.chain(*[list(bigrams(s)) for s in filtered_corpus]))
    vocab = list(set(itertools.chain(*filtered_corpus)))
    vocab_index = {word: i for i, word in enumerate(vocab)}

    matrix = np.zeros((len(vocab), len(vocab)))
    for bg in bi_grams:
        i = vocab_index.get(bg[1])
        j = vocab_index.get(bg[0])
        if i is not None and j is not None:
            matrix[i][j] += 1
            matrix[j][i] += 1  # symmetric

    return matrix, vocab_index


tweets = corpus['Tweet'].tolist()
matrix, vocab_index = generate_co_occurrence_matrix(tweets)
data_matrix = pd.DataFrame(matrix, index=vocab_index, columns=vocab_index)
print(f'Vocabulary size: {len(vocab_index)}')
print(f'Matrix shape: {data_matrix.shape}')


> **Output interpretation:** The vocabulary grows with corpus size. A large vocabulary produces a very sparse matrix — most word pairs never co-occur as bigrams. The diagonal is zero (a word cannot be its own left/right neighbour in a bigram). Words that frequently appear together will have high counts.


---

## 4. Heatmap Visualisation

The heatmap encodes co-occurrence frequency as colour intensity.  
Brighter cells indicate word pairs that appear together more frequently in the corpus.


In [ ]:
plt.figure(figsize=(14, 7))
sns.heatmap(data_matrix, cmap='YlOrRd')
plt.title('Word Co-occurrence Matrix — Heatmap')
plt.tight_layout()
plt.show()


> **Output interpretation:** High-frequency word pairs (bright cells) cluster around common topic words in the corpus. Most of the matrix is near-zero (dark blue), confirming that co-occurrence matrices are extremely sparse. Words that appear rarely will show mostly dark rows and columns.


---

## 5. Clustermap Visualisation

A **clustermap** applies hierarchical clustering to both rows and columns, grouping  
semantically related words. Words that share similar co-occurrence profiles end up  
next to each other in the reordered matrix.


In [ ]:
plt.figure(figsize=(14, 7))
sns.clustermap(data_matrix, cmap='YlOrRd', figsize=(14, 7))
plt.suptitle('Word Co-occurrence Matrix — Clustermap', y=1.02)
plt.show()


> **Output interpretation:** Hierarchical clustering reorders the matrix rows and columns so that words with similar distributional profiles (co-occur with the same neighbours) are grouped. Dense blocks along the diagonal represent semantic clusters — groups of words that tend to appear in the same contexts.


---

## Summary

| Step | Tool | Purpose |
|---|---|---|
| Tokenisation | NLTK `word_tokenize` | Split tweets into word tokens |
| Filtering | Spanish stopwords + min length | Remove noise words |
| Bigrams | NLTK `bigrams` | Capture adjacent word pairs |
| Matrix | NumPy zeros + increment | Count symmetric co-occurrences |
| Heatmap | Seaborn | Visualise raw frequency |
| Clustermap | Seaborn | Group semantically similar words |
